In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
# Loading in the adult dataset
from google.colab import files
uploaded = files.upload()
adult = pd.read_csv("adult.csv")

Saving adult.csv to adult.csv


In [4]:
# Cleaning workclass
mapping = {
    "Never-worked":"Other",
    "Without-pay":"Other",
    "?":"Other"
    }
adult["workclass"] = adult["workclass"].replace(mapping)

In [5]:
# Cleaning education
mapping = {
    "Preschool":"Pre-High School",
    "1st-4th":"Pre-High School",
    "5th-6th":"Pre-High School",
    "7th-8th":"Pre-High School",
    "9th":"Pre-High School",
    "10th":"Some High School",
    "11th":"Some High School",
    "12th":"Some High School",
    "Prof-school":"Prof-school/Doctorate",
    "Doctorate":"Prof-school/Doctorate"
    }
adult["education"] = adult["education"].replace(mapping)

In [6]:
# Cleaning marital-status
adult = adult[adult["marital-status"]!="Married-AF-spouse"]

In [7]:
# Cleaning occupation type
mapping = {
    "Armed-Forces":"Unknown or other",
    "Priv-house-serv":"Unknown or other",
    "?":"Unknown or other",
    }
adult["occupation"] = adult["occupation"].replace(mapping)

In [8]:
# Cleaning sex
adult["sex:female"] = np.where(adult["sex"]=="Female", 1, 0)

In [9]:
# Cleaning native-country
adult["native-country:US"] = np.where(adult["native-country"]=="United-States", 1, 0)

In [10]:
# Cleaning income
adult["income"] = adult["income"].str.replace('.','')
adult["income>50K"] = np.where(adult["income"]==">50K", 1, 0)

In [11]:
# Replacing missing values
adult = adult.fillna({"occupation": "Unknown or other", "workclass": "Other"})

In [12]:
# Selecting only relevant variables
adult = adult[["age", "workclass", "education", "marital-status", "occupation",
               "relationship", "sex:female", "hours-per-week", "native-country:US", "income>50K"]]

In [13]:
# 1. Separate features (X) and target (y)
X = adult.drop("income>50K", axis=1)
y = adult["income>50K"]

In [14]:
# 2. Create dummy variables
cols_to_dummy = ["workclass", "education", "marital-status", "occupation", "relationship"]
X = pd.get_dummies(X, columns=cols_to_dummy)

In [15]:
# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# 4. Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
# Convert y to numpy arrays to ensure smooth processing with Keras (X is already converted by the scaler)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [18]:
# 5. Build the Neural Network Classifier
model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [19]:
# Compile the model using binary crossentropy
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

In [20]:
# 6. Train the model
history = model.fit(X_train_scaled,
                    y_train,
                    epochs=20,
                    batch_size=32,
                    validation_split=0.2)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7987 - loss: 0.4056 - val_accuracy: 0.8394 - val_loss: 0.3404
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8393 - loss: 0.3506 - val_accuracy: 0.8433 - val_loss: 0.3343
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8401 - loss: 0.3473 - val_accuracy: 0.8457 - val_loss: 0.3337
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8417 - loss: 0.3451 - val_accuracy: 0.8443 - val_loss: 0.3344
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8426 - loss: 0.3438 - val_accuracy: 0.8438 - val_loss: 0.3329
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8429 - loss: 0.3427 - val_accuracy: 0.8468 - val_loss: 0.3320
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8432 - loss: 0.3421 - val_accuracy: 0.8468 - val_loss: 0.3322
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8451 - loss: 0.3411 - val_accuracy: 0.

In [21]:
# 7. Evaluate the model
test_loss, test_acc = model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8306 - loss: 0.3529
Test Accuracy: 0.8306
